[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/kane_mele/kane_mele.ipynb)

# Edge States of a Graphene Flake

This example computes the electronic spectrum and helical edge transport of a finite graphene flake with intrinsic spin-orbit coupling, introduced by Kane and Mele (2005). Spin-orbit coupling opens a bulk energy gap across the interior of the honeycomb flake while supporting gapless states along its rim, where opposite spins circulate in opposite senses. Adding a staggered sublattice potential, as in hexagonal boron nitride, competes with that gap and drives the flake into an ordinary insulator once the staggered mass exceeds the bulk spin-orbit scale.

Each atom's state is modelled as an even multivector spinor, and the flake's Hamiltonian couples neighbouring sites through multivectors: minus one on nearest-neighbour bonds, the spin-orbit strength times the unit bivector `(near - between).commutator(between - far).normalized()` of the plane a second-neighbour hop turns in, and the sublattice mass. Multiplying the Hamiltonian on the right by `Up` restricts its domain to spin-up spinors in the `xy` plane and closes within `Up` automatically by type inference. Each eigenstate advances in time through right-hand phase rotation `(xy * (-energies * time)).exp()`, and sandwiching the intrinsic spin patch `up_spin` via `state >> up_spin` gives the atomic spin density.

In second-quantized Pauli-spinor notation, with a two-component complex column $c_i = (c_{i\uparrow}, c_{i\downarrow})^{\mathsf{T}}$ on each site and manually assigned sign factors $\nu_{ij} = (\mathbf{d}_1 \times \mathbf{d}_2)_z / |\mathbf{d}_1 \times \mathbf{d}_2| = \pm 1$ for left and right second-neighbour turns, the Hamiltonian splits into nearest-neighbour, Haldane spin-orbit, and staggered Semenoff terms as $H = -t\sum_{\langle ij\rangle} c_i^\dagger c_j + i\lambda_{\mathrm{SO}}\sum_{\langle\langle ij\rangle\rangle} \nu_{ij}\, c_i^\dagger s_z c_j + m\sum_i \xi_i\, c_i^\dagger c_i$, with time evolution $\psi_n(t) = e^{-i E_n t}\psi_n(0)$ and spin expectation $\psi_i^\dagger \tfrac{1}{2}\boldsymbol{\sigma}\psi_i$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import VGA3D
from numga.gatype import GAType
from numga.sparse import Sparse, SparseExtensor, spdiag
from numga.topology import Topology
from examples.util.animation import save_figure, show_states
from examples.quantum.kane_mele import render, scenarios

np.set_printoptions(precision=3, suppress=True)

ga = VGA3D
mv = NumpyContext(ga).multivector
# + extrinsic, - intrinsic
Scalar = ga.gatype.scalar()                                                    # Scalar+
Spin = ga.gatype.antibivector()                                                # Antibivector-
Point = ga.gatype.antivector()                                                 # Antivector-
Even = ga.gatype.even()                                                        # Even+
# Spin-up spinors lie in the xy plane; spin-down spinors are bivector planes touching z.
Up = Even.derive.within(Algebra("x+y+"))                                       # Even+
Down = Even.derive.touching(Algebra("z+"))                                     # Even+
up_spin = mv.xy.dual()                                                         # [] Spin-

## Multivector tight-binding Hamiltonian

Each coupled pair of carbon atoms carries a multivector weight that multiplies an input atom's spinor into the output atom's spinor. On a second-neighbour bond, the electron steps through an intermediate site of the opposite sublattice, and the commutator of the two bond steps spans the unit bivector of the turn plane. Because the bivector's orientation records whether the two-step path bends left or right, mirroring the flake mirrors the spin-orbit coupling automatically without manual sign tables.

Multiplying the Hamiltonian on the right by `Up` restricts its input to even multivectors in the `xy` plane, and because every coupling is a scalar or an `xy` bivector, the output stays in `Up` while `Down` likewise maps planes touching `z` into themselves.

In [ ]:
flake = scenarios.honeycomb_flake(scenarios.flake_size, scenarios.rim_fraction)
spin_orbit = scenarios.spin_orbit
gap_threshold = scenarios.gap_threshold
save_figure(render.flake_figure(flake), "kane_mele_flake");

Blue and terracotta dots mark the two triangular sublattices of the armchair-edged honeycomb flake, joined by grey nearest-neighbour bonds.

In [ ]:
def hamiltonian(bonds: Topology, next_bonds: Topology, positions: Point, between: Point,
                sublattice: Scalar, spin_orbit: float, mass: float) -> SparseExtensor:
    """Nearest-neighbour hops, second-neighbour turn planes, and the staggered sublattice mass sum
    into the flake's sparse multivector Hamiltonian (`Up+[V <- V]`)."""
    ends = bonds.incidence[1, 0]                                               # [2] Scalar+[E <- V]
    forward_hop = -(ends[0].adjugate() * ends[1])                              # Scalar+[V <- V]
    # A second-neighbour hop steps from the second endpoint through the intermediate lattice site
    # to the first; the commutator of its two steps gives the oriented plane of the turn.
    next_ends = next_bonds.incidence[1, 0]                                     # [2] Scalar+[E_next <- V]
    near, far = next_ends * positions                                          # Point-[E_next], Point-[E_next]
    turns = (near - between).commutator(between - far).normalized()            # Bivector+[E_next]
    forward_turn = next_ends[0].adjugate() * spdiag(turns) * next_ends[1]      # Bivector+[V <- V]
    return (forward_hop + forward_hop.adjugate()
            + (forward_turn + ~forward_turn) * spin_orbit + sublattice * mass)   # Up+[V <- V]


zero_mass_energy = hamiltonian(
    flake.bonds, flake.next_bonds, flake.positions, flake.between, flake.sublattice, spin_orbit, 0.0,
)                                                                              # Up+[V <- V]
print("couplings:       ", zero_mass_energy.gatype.output_subspace)
print("spin up, out/in: ", *(zero_mass_energy * Up).gatype.axes)
print("spin down, out/in:", *(zero_mass_energy * Down).gatype.axes)

## Energy levels across the topological transition

Restricting the Hamiltonian to `Up` produces a Hermitian endomorphism on spin-up fields whose eigenpairs nearest zero track the closing and reopening of the bulk gap as the staggered sublattice mass increases.

In [ ]:
n_masses = scenarios.n_masses
masses = scenarios.masses
n_levels = scenarios.n_levels

In [ ]:
def sector_levels(energy: SparseExtensor, sector: GAType, n_levels: int) -> tuple[Scalar, Even]:
    """The `n_levels` eigenpairs nearest zero energy within `sector` give energies (`[n_levels] Scalar+`)
    and orthonormal spinor fields (`[n_levels] Even+[V]`)."""
    V = energy.shape[0]
    unit = Sparse.selection(np.arange(V), V) * sector                          # (Even+ <- Even+)[V <- V]
    return (energy * sector).eigh(unit, n_levels)                              # [n_levels] Scalar+, [n_levels] Even+[V]


def rim_share(rim: Scalar, states: Even) -> Scalar:
    """The fraction of each state's norm squared lying on the rim atoms (`[...] Scalar+`)."""
    density = states.scalar_norm_squared()                                     # [...] Scalar+[V]
    return (density * rim).sites.sum() / density.sites.sum()                   # [...] Scalar+


spectra = [
    sector_levels(
        hamiltonian(flake.bonds, flake.next_bonds, flake.positions, flake.between,
                    flake.sublattice, spin_orbit, float(mass)),
        Up, n_levels,
    )
    for mass in masses
]
energies = stack([level for level, _ in spectra])                              # [n_masses, n_levels] Scalar+
shares = stack([rim_share(flake.rim, states) for _, states in spectra])        # [n_masses, n_levels] Scalar+
save_figure(render.levels_figure(masses, energies, shares, gap_threshold), "kane_mele_levels");

Yellow dots mark states localized on the flake's outer rim and purple dots mark interior bulk states, while the vertical dashed line marks the critical staggered mass where the bulk gap closes and reopens without in-gap edge branches.

## Eigenstates below and above the gap closing

Comparing the spin-up eigenstate nearest zero energy at zero staggered mass against the state well above the transition shows how the lowest mode shifts from the boundary into the bulk.

In [ ]:
n_cases = scenarios.n_cases
sample_masses = scenarios.sample_masses

In [ ]:
sample_states = stack([
    sector_levels(
        hamiltonian(flake.bonds, flake.next_bonds, flake.positions, flake.between,
                    flake.sublattice, spin_orbit, float(mass)),
        Up, 2,
    )[1][0]
    for mass in sample_masses
])                                                                             # [n_cases] Up+[V]
densities = sample_states.scalar_norm_squared()                                # [n_cases] Scalar+[V]
save_figure(render.states_figure(flake, densities), "kane_mele_states");

In the left panel at zero mass, the state nearest zero forms a tight ring along the armchair rim; in the right panel above the transition, the lowest state spreads across the interior of the flake. Marker area and darkness both scale with the state's probability density on each atom.

## Helical spin transport along the rim

Starting an electron on a single atom on the right-hand edge with its spin pointing along `x` creates an equal superposition of spin-up and spin-down components, and projecting onto the in-gap eigenstates isolates the helical boundary modes.

In [ ]:
n_in_gap = scenarios.n_in_gap
initial_spinor = scenarios.initial_spinor
n_frames = scenarios.n_frames
times = scenarios.times
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def spin_evolution(states: Even, energies: Scalar, start_site: Scalar, spinor: Even,
                   times: np.ndarray) -> Iterator[Spin]:
    """Yields the atomic spin density (`Spin-[V]`) at each step in `times` for an electron started
    on `start_site` with `spinor` and projected onto `states`."""
    weights = (states * start_site).sites.sum().reverse().scalar_product(spinor)   # [...] Scalar+
    for time in times:
        # Each eigenstate turns on its right in the xy plane at the rate of its energy.
        phases = (mv.xy * (-energies * time)).exp()                            # [...] Even+
        state = (states * (weights * phases)).sum(axis=0)                      # Even+[V]
        yield state >> up_spin                                                 # Spin-[V]


in_gap, edge_states = sector_levels(zero_mass_energy, Even, n_in_gap)          # [n_in_gap] Scalar+, [n_in_gap] Even+[V]
spin_frames = show_states(
    spin_evolution(edge_states, in_gap, flake.start_site, initial_spinor, times),
    render.spin_animation, flake, render.flake_pixels,
    name="kane_mele_helical", duration_ms=frame_duration_ms, n_frames=n_frames + 1,
);

Marker area shows the electron's probability density on each atom, coloured red where spin points up (`+z`) and blue where spin points down (`-z`): the red spin-up wavepacket travels counterclockwise around the armchair boundary while the blue spin-down wavepacket travels clockwise.

In [ ]:
# checks
spins = stack(spin_frames)                                                     # [n_frames + 1] Spin-[V]
# The Hamiltonian equals its own reverse because every coupling reverses into the opposite direction's coupling.
assert (((~zero_mass_energy - zero_mass_energy) * edge_states).scalar_norm_squared().sites.sum() < 1e-20).all()
# At zero mass the levels inside the bulk gap sit on the rim; well past the transition the gap is empty.
half_gaps = np.abs(gap_threshold - masses)
inside_gap = energies[0].abs() < 0.9 * half_gaps[0]
assert inside_gap.sum() >= 8 and (shares[0][inside_gap] > 0.9).all()
assert energies[-1].abs().min() > 0.5 * half_gaps[-1]
# The state nearest zero sits on the rim below the transition and spreads into the interior above it.
case_shares = rim_share(flake.rim, sample_states)                              # [n_cases] Scalar+
assert bool(case_shares[0] > 0.9) and bool(case_shares[-1] < 0.6)
# Total edge-projected weight is conserved and stays localized on the rim throughout the circuit.
density = spins.norm()                                                         # [n_frames + 1] Scalar+[V]
totals = density.sites.sum()                                                   # [n_frames + 1] Scalar+
assert ((totals - totals[0]).abs() < 1e-9 * totals[0]).all()
assert ((density * flake.rim).sites.sum() > 0.9 * totals).all()
# A quarter lap in, the spin-up packet has moved counterclockwise and the spin-down packet clockwise.
quarter = n_frames // 4
spin_z = mv.xy & spins[quarter]                                                # Scalar+[V]
up_half = (density[quarter] + spin_z) * 0.5                                    # Scalar+[V]
down_half = (density[quarter] - spin_z) * 0.5                                  # Scalar+[V]
start_point = (flake.positions * flake.start_site).sites.sum().normalized()    # [] Point-
up_centre = ((up_half * flake.positions).sites.sum() / up_half.sites.sum()).normalized()     # [] Point-
down_centre = ((down_half * flake.positions).sites.sum() / down_half.sites.sum()).normalized()   # [] Point-
assert mv.xy & (start_point & up_centre) > 0.5
assert mv.xy & (start_point & down_centre) < -0.5